In [ ]:
# Import our dependecies
import pandas as pd
import numpy as np
import hvplot.pandas
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from warnings import simplefilter
simplefilter(action='ignore')

In [ ]:
# Read in the data
df = pd.read_csv("../Data/AdultCensusUpdated.csv")
census_df = df.copy()
df.head()

In [ ]:
# Clean excessive column
df = df.drop(['fnlwgt','education','capital.gain','capital.loss','native.country'], axis=1)
# Replace null values
df = df.replace("?", np.nan)
df = df.fillna(0)
df.head()

In [ ]:
# Manual binary encoding for specified columns
df["income"] = df["income"].map({"<=50K": 0, ">50K": 1})
df["sex"] = df["sex"].map({"Male": 0, "Female": 1})
df.head()

In [ ]:
# Create a LabelEncoder instance
label_encoder = LabelEncoder()

# Apply Label Encoding to each non-numeric column
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = label_encoder.fit_transform(df[col].astype(str))
df.head()

In [ ]:
# Confirm Data Types
df.dtypes

## PART 2: Creating the Elbow Curve

In [ ]:
# Identify target column and separate from main dataframe 
y = df['income']
X = df.drop(columns='income')

In [ ]:
# Split data into training and testing groups
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

In [ ]:
# Create a a list to store inertia values
inertia = []

# Create a a list to store the values of k
k = list(range(1, 11))

# Create a for loop where each value of k is evaluated using the K-means algorithm
# Fit the model using the home_sales_df DataFrame
# Append the value of the computed inertia from the `inertia_` attribute of the K-means model instance
for i in k:
    k_model = KMeans(n_clusters=i, random_state=1)
    k_model.fit(X)
    inertia.append(k_model.inertia_)

# Create a dictionary that holds the list values for k and inertia
elbow_data = {"k": k, "inertia": inertia}

# Create a DataFrame using the elbow_data dictionary
df_elbow = pd.DataFrame(elbow_data)

In [ ]:
# Plot the DataFrame
df_elbow.hvplot.line(
    x="k", 
    y="inertia", 
    title="Elbow Curve", 
    xticks=k
)

## Part 3: Instantiate an K Nearest Neighbor Classifier instance

In [ ]:
# Instantiate a StandardScaler instance
scaler = StandardScaler()

# Fit the training data to the standard scaler
X_scaler = scaler.fit(X_train)

# Transform the training data using the scaler
X_train_scaled = X_scaler.transform(X_train)

# Transform the testing data using the scaler
X_test_scaled = X_scaler.transform(X_test)

In [ ]:
# Instantiate the KNeighborsClassifier model with n_neighbors = 3 
knn = KNeighborsClassifier(n_neighbors=3)

In [ ]:
# Train the model using the training data
knn.fit(X_train_scaled, y_train)

In [ ]:
# Create predictions using the testing data
y_pred = knn.predict(X_test_scaled)

In [ ]:
# Print the classification report comparing the testing data to the model predictions
print(classification_report(y_test, y_pred))

In [ ]:
# Reference column names
X.columns

## Part 4: Model Optimization

In [ ]:
# Scale the data
data_scaled = StandardScaler().fit_transform(X[['age', 'workclass', 'education.num', 'marital.status', \
       'occupation', 'relationship', 'race', 'sex', \
        'hours.per.week','State']])


In [ ]:
# Create a DataFrame called with the scaled data
df_scaled = pd.DataFrame(
    data_scaled,
    columns=['age', 'workclass',  'education.num', 'marital.status', \
       'occupation', 'relationship', 'race', 'sex', 'hours.per.week', 'State']
)
X = df_scaled
# Review the DataFrame
X.head()

In [ ]:
# Split into training and testing data with new scaled data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

In [ ]:
# Instantiate a StandardScaler instance
scaler = StandardScaler()

# Fit the training data to the standard scaler
X_scaler = scaler.fit(X_train)

# Transform the training data using the scaler
X_train_scaled = X_scaler.transform(X_train)

# Transform the testing data using the scaler
X_test_scaled = X_scaler.transform(X_test)

In [ ]:
# Instantiate the KNeighborsClassifier model with n_neighbors = 3 
knn = KNeighborsClassifier(n_neighbors=3)

In [ ]:
# Train the model using the training data
knn.fit(X_train_scaled, y_train)

In [ ]:
# Create predictions using the testing data
y_pred = knn.predict(X_test_scaled)

In [ ]:
# Print the classification report comparing the testing data to the model predictions
print(classification_report(y_test, y_pred))

In [ ]:
print(y_pred)

In [ ]:
# Add model name to dataframe
model_name = []
for value in y_pred:
    model_name.append("KNeighbors")
    

In [ ]:
# Add predictions and model to dataframe
X_test['Results'] = y_pred
X_test['Model'] = model_name

In [ ]:
# Verify Results
X_test.head()

In [ ]:
X_test.columns

In [ ]:
# Drop encoded data
X_test = X_test.drop(columns =['age', 'workclass', 'education.num', 'marital.status', 'occupation',
       'relationship', 'race', 'sex', 'hours.per.week', 'State'])

In [ ]:
# Merge with original dataframe
new_df = census_df.join(X_test)

In [ ]:
# Drop training data
new_df = new_df.dropna()

In [ ]:
# Verify Results
new_df.head()

In [ ]:
# Export to CSV
new_df.to_csv("../Data/KNeighbors.csv")